<a href="https://colab.research.google.com/github/JiviteshG/GenerativeAI/blob/main/First_Bot_program.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Hugging face transformers library
Dialup GPT for conversation model from building ChatBots

In [48]:
!pip install transformers

In [73]:
from transformers import AutoModelForCausalLM, AutoTokenizer # Loads pre trained model for Casual Language Modeling used for Text Generation task # AutoTokenizer converts text into tokens
import torch # Deep learning framework used by the model which rely on these tensors to make the model work

In [74]:
model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)  # DialoGPT is trained on larger database of dialogues which enables user to generate conversational responses
model = AutoModelForCausalLM.from_pretrained(model_name)

In [75]:
chat_history_ids = None # Store the history of conversation to store contect of the conversation
print("Start chatting with your AI! Type 'quit' to exit\n")

Start chatting with your AI! Type 'quit' to exit



In [77]:
# Main logic of chat bot
while True:
    user_input = input("You: ")
    if user_input.lower() in ['exit', 'quit', 'bye']:
        break

    # Encode user input and append end-of-sentence token
    # user_input is converted ino tokens using the tokenizer.encode.
    # So eos_token (end of sequence token) is added at the end of user input.
    # This tels the model that the input has ended and it prepares for generating a reponse
    # return_tensors = pt makes sure that the encoded response id returened as a pytorch tensor
    #  This the format the model expects
    new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Append new user input to chat history
    # Keeping a track of the chat history so the model can respond - chat_history_ids
    bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1) if chat_history_ids is not None else new_user_input_ids

    # Generate response with sampling for creativity
    # model.generate: This line sens cancatenated response to tyhe model
    # max_length: max length of 1000 tokens in poutput
    # pad_token_id: model needs to know which tokens are padding tokens
    #
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=2000,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
        do_sample=True,
        top_k=50,
        top_p=0.95
    )

    # Decode and print only the new response tokens
    # tokenizer.decode: The ouput of model is converted nack into huma readable response
    # skip_special_tokens: the special tokens like eos are not included in the output
    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

    # bot response is displayed on the screes
    # the context is preserved
    print(f"Bot: {response}")

You: Explain Generative AI
Bot: Why do you even need an AI?
You: To automate tasks, isn't it?
Bot: But why would we need an automated robot?
You: To make hums work less I guess.
Bot: They're also a way for humans to work.
You: exit


## FIN